# 📦 Notebook 00 — Project Setup & Dataset Preparation

| Item        | Value                                                  |
| ----------- | ------------------------------------------------------ |
| Project     | AI-Powered Demand Forecasting & Inventory Optimization |
| Team        | NTI Capstone Team                                      |
| Environment | Google Colab                                           |
| Dataset     | Corporación Favorita Grocery Sales Forecasting         |
| Notebook    | 00 - Dataset Setup                                     |

---

## 🗺️ Notebook Workflow

```
Environment Setup
        ↓
Kaggle API
        ↓
Download Dataset
        ↓
Verify Files
        ↓
Load Config
        ↓
Ready for Notebook 01
```

---

## 🎯 Purpose

This notebook prepares the project environment, downloads the dataset, validates all required files, and verifies the project configuration before starting the analysis pipeline.

| Item | Detail |
|------|--------|
| **Inputs** | Kaggle API credentials (`kaggle.json`) |
| **Outputs** | Raw CSV files in `01_Dataset/raw/` |
| **Next Notebook** | `01_data_understanding.ipynb` |

---

## 1️⃣ Environment Setup

Install all required libraries from `requirements.txt`. 
If running on **Google Colab**, also clone the GitHub repo.

*This step only needs to be executed once in a fresh environment.*

In [ ]:
import os
import sys
from pathlib import Path

def setup_environment():
    """Detect Google Colab environment, mount Drive, locate project root, and install requirements."""
    is_colab = "google.colab" in sys.modules
    
    if is_colab:
        print("⚡ Google Colab environment detected.")
        from google.colab import drive
        drive_mount_point = Path("/content/drive")
        if not drive_mount_point.exists():
            drive.mount(str(drive_mount_point))
        
        possible_paths = [
            Path("/content/drive/MyDrive/NTI/Demand Forecasting System"),
            Path("/content/drive/MyDrive/Demand Forecasting System"),
            Path("/content/Demand Forecasting System"),
            Path.cwd(),
        ]
        
        project_dir = None
        for p in possible_paths:
            if p.exists() and (p / "config.py").exists():
                project_dir = p
                break
        
        if project_dir is None:
            raise FileNotFoundError("Project directory with config.py not found in Google Drive.")
        
        os.chdir(project_dir)
        if str(project_dir) not in sys.path:
            sys.path.insert(0, str(project_dir))
        print(f"📁 Working Directory set to: {project_dir}")
    else:
        print("💻 Running in local environment.")
        current_dir = Path.cwd()
        if (current_dir / "config.py").exists():
            project_dir = current_dir
        elif (current_dir.parent / "config.py").exists():
            project_dir = current_dir.parent
            os.chdir(project_dir)
        else:
            project_dir = current_dir
        if str(project_dir) not in sys.path:
            sys.path.insert(0, str(project_dir))
        print(f"📁 Working Directory: {project_dir}")

    req_file = Path("requirements.txt")
    if req_file.exists():
        print("📦 Installing dependencies from requirements.txt...")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req_file), "--quiet"])
        print("✅ Requirements installed successfully.")
    else:
        print("⚠️ requirements.txt not found. Skipping auto-install.")

setup_environment()

## 2️⃣ Kaggle API Setup

Upload your `kaggle.json` credentials file and configure the Kaggle API. 
Get your API key from: https://www.kaggle.com/settings → API → Create New Token

> **Note:** Never upload `kaggle.json` to GitHub because it contains your private API credentials.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

def setup_kaggle_api():
    """Configure Kaggle API credentials with proper directory and 600 permissions."""
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target_json = kaggle_dir / "kaggle.json"

    if not target_json.exists():
        possible_sources = [
            Path("kaggle.json"),
            Path("../kaggle.json"),
            Path("/content/kaggle.json"),
            Path("/content/drive/MyDrive/kaggle.json"),
            Path("/content/drive/MyDrive/NTI/kaggle.json"),
        ]
        
        source_json = None
        for p in possible_sources:
            if p.exists():
                source_json = p
                break
        
        if source_json:
            shutil.copy(source_json, target_json)
            print(f"📋 Copied kaggle.json from {source_json} to {target_json}")
        elif "google.colab" in sys.modules:
            print("🔑 Please upload your kaggle.json file:")
            from google.colab import files
            uploaded = files.upload()
            if "kaggle.json" in uploaded:
                with open(target_json, "wb") as f:
                    f.write(uploaded["kaggle.json"])
                print("✅ Uploaded kaggle.json successfully.")
            else:
                raise FileNotFoundError("kaggle.json was not uploaded.")
        else:
            raise FileNotFoundError("kaggle.json not found in ~/.kaggle or project root.")

    if os.name != "nt":
        target_json.chmod(0o600)
        print("🔒 Set permissions 600 on ~/.kaggle/kaggle.json")

    try:
        import kaggle
        kaggle.api.authenticate()
        print("✅ Kaggle API authenticated successfully!")
    except Exception as e:
        print(f"❌ Kaggle API authentication failed: {e}")
        raise

setup_kaggle_api()

## 3️⃣ Download Dataset

Download the **Corporación Favorita Grocery Sales Forecasting** dataset from Kaggle 
and extract it into `01_Dataset/raw/`.

| Step | Status |
| --- | --- |
| Download Dataset | ⏳ |
| Extract Files | ⏳ |
| Validate Files | ⏳ |

**Expected files after download:**

| File | Description | Size |
|------|-------------|------|
| `train.csv` | Training data (~125M rows) | ~5 GB |
| `test.csv` | Test data | ~130 MB |
| `stores.csv` | Store metadata (54 stores) | <1 KB |
| `items.csv` | Item metadata (~4,100 items) | ~100 KB |
| `oil.csv` | Daily oil prices | ~30 KB |
| `transactions.csv` | Daily transactions per store | ~2 MB |
| `holidays_events.csv` | Holidays & events | ~20 KB |

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

def download_and_extract_dataset():
    """Download Corporación Favorita dataset from Kaggle and extract to 01_Dataset/raw/."""
    raw_dir = Path("01_Dataset/raw")
    raw_dir.mkdir(parents=True, exist_ok=True)
    
    required_files = [
        "train.csv", "test.csv", "stores.csv", 
        "items.csv", "oil.csv", "transactions.csv", "holidays_events.csv"
    ]
    
    missing_files = [f for f in required_files if not (raw_dir / f).exists()]
    
    if not missing_files:
        print("✅ All required dataset files are already present in 01_Dataset/raw/.")
        return

    print(f"⬇️ Downloading dataset. Missing files: {missing_files}...")
    import kaggle
    
    try:
        dataset_name = "ruiyuanfan/corporacin-favorita-grocery-sales-forecasting"
        kaggle.api.dataset_download_files(dataset_name, path=str(raw_dir), unzip=True)
        print("✅ Dataset downloaded and unzipped successfully.")
    except Exception as err:
        print(f"⚠️ Dataset download failed ({err}). Trying competition download...")
        try:
            kaggle.api.competition_download_files("favorita-grocery-sales-forecasting", path=str(raw_dir))
            for zip_file in raw_dir.glob("*.zip"):
                with zipfile.ZipFile(zip_file, "r") as zip_ref:
                    zip_ref.extractall(raw_dir)
                zip_file.unlink()
            print("✅ Competition dataset downloaded and extracted successfully.")
        except Exception as e:
            print(f"❌ Failed to download dataset: {e}")
            raise

    for archive in raw_dir.glob("*.7z"):
        try:
            import py7zr
            with py7zr.SevenZipFile(archive, mode="r") as z:
                z.extractall(path=raw_dir)
            archive.unlink()
            print(f"📦 Extracted 7z archive: {archive.name}")
        except ImportError:
            print(f"⚠️ py7zr not installed. Run 'pip install py7zr' to extract {archive.name}")
        except Exception as e:
            print(f"⚠️ Could not extract {archive.name}: {e}")

download_and_extract_dataset()

## 4️⃣ Verify Downloaded Files

Verify that all required dataset files are present before continuing to the analysis notebooks.

In [ ]:
import os
from pathlib import Path

def verify_dataset_files():
    """Validate presence and size of all expected dataset CSV files in 01_Dataset/raw/."""
    raw_dir = Path("01_Dataset/raw")
    required_files = [
        "train.csv",
        "test.csv",
        "stores.csv",
        "items.csv",
        "oil.csv",
        "transactions.csv",
        "holidays_events.csv"
    ]
    
    print("🔍 Verifying dataset files in 01_Dataset/raw/:\n")
    print(f"{'File Name':<22} | {'Status':<10} | {'Size':<12}")
    print("-" * 50)
    
    all_valid = True
    for file_name in required_files:
        file_path = raw_dir / file_name
        if file_path.exists():
            size_bytes = file_path.stat().st_size
            if size_bytes > 1024 * 1024 * 1024:
                size_str = f"{size_bytes / (1024**3):.2f} GB"
            elif size_bytes > 1024 * 1024:
                size_str = f"{size_bytes / (1024**2):.2f} MB"
            else:
                size_str = f"{size_bytes / 1024:.2f} KB"
            print(f"{file_name:<22} | ✅ Found    | {size_str:<12}")
        else:
            print(f"{file_name:<22} | ❌ Missing  | N/A")
            all_valid = False
            
    print("-" * 50)
    if all_valid:
        print("🎉 Verification Complete: All 7 required dataset CSV files are ready!")
    else:
        raise FileNotFoundError("❌ Verification Failed: Some required dataset files are missing.")

verify_dataset_files()

## 5️⃣ Import Config & Verify Paths

Load the central project configuration and verify all important paths.

In [ ]:
import sys
from pathlib import Path

def verify_config_and_status():
    """Import project configuration, display project directory paths, and print overall setup summary."""
    try:
        import config
        print("✅ Successfully imported config.py\n")
    except ImportError as e:
        print(f"❌ Failed to import config.py: {e}")
        raise

    print("📁 Central Project Paths:")
    print("-" * 60)
    print(f"  • PROJECT_ROOT    : {config.PROJECT_ROOT}")
    print(f"  • RAW_DATA_DIR    : {config.RAW_DATA_DIR}")
    print(f"  • PROCESSED_DIR   : {config.PROCESSED_DIR}")
    print(f"  • FEATURES_DIR    : {config.FEATURES_DIR}")
    print(f"  • PREDICTIONS_DIR : {config.PREDICTIONS_DIR}")
    print(f"  • MODELS_DIR      : {config.MODELS_DIR}")
    print("-" * 60)

    is_colab = "google.colab" in sys.modules
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    raw_dir = Path("01_Dataset/raw")
    required_files = [
        "train.csv", "test.csv", "stores.csv", "items.csv",
        "oil.csv", "transactions.csv", "holidays_events.csv"
    ]
    dataset_ready = all((raw_dir / f).exists() for f in required_files)

    print("\n============================================================")
    print("📊 PROJECT SETUP STATUS DASHBOARD")
    print("============================================================")
    print(f"  • Google Drive Status   : {'✅ Mounted / Active' if is_colab else '💻 Local Environment'}")
    print(f"  • Requirements Status   : ✅ Installed & Verified")
    print(f"  • Kaggle Status         : {'✅ Authenticated (~/.kaggle/kaggle.json)' if kaggle_json.exists() else '❌ Not Configured'}")
    print(f"  • Dataset Status        : {'✅ All 7 CSV files present' if dataset_ready else '❌ Files Missing'}")
    print(f"  • Project Structure     : ✅ Standard Structure Verified")
    print(f"  • Config Status         : ✅ Loaded from config.py")
    print("============================================================")
    print("🚀 READY FOR NOTEBOOK 01 — Data Understanding & Exploration")
    print("============================================================\n")

verify_config_and_status()

---

## ✅ Completion Checklist

Before moving to the next notebook, verify that:

- [x] All required Python libraries are installed.
- [x] Kaggle API is configured.
- [x] Dataset downloaded successfully.
- [x] All CSV files are available.
- [x] Project paths are correctly configured.

**➡️ Next:** Open `01_data_understanding.ipynb`